# AI Model Experiment & Evaluation
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membandingkan dua pendekatan AI dalam menyelesaikan task sentiment analysis ulasan pelanggan:
1. Model klasik Machine Learning (Scikit-learn)
2. LLM API (Gemini)

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

**Konteks.** Tim produk e-commerce ingin mengklasifikasikan sentimen ulasan pelanggan secara otomatis pada halaman produk. Sebelum memilih pendekatan untuk produksi, dilakukan eksperimen awal yang membandingkan dua kandidat.

**Objective.** Membandingkan (1) model klasik Machine Learning (TF-IDF + Scikit-learn) yang dilatih sendiri dan (2) LLM Gemini via API dengan prompting tanpa training, pada test set yang sama, menggunakan Accuracy, Precision, Recall, F1-Score, dan confusion matrix, lalu menyusun rekomendasi technical approach berbasis hasil eksperimen.

**Target/label.** Kolom `sentiment` dengan dua kelas: `positif` dan `negatif` (klasifikasi biner). Input yang digunakan hanya kolom `review_text`.

**Batasan dan asumsi.**
- Dataset resmi dari instruktur digunakan apa adanya dan tidak dimodifikasi.
- Label pada dataset dianggap sebagai ground truth.
- `product_name` tidak dipakai sebagai fitur, karena sentimen ditentukan oleh isi ulasan, bukan nama produk.
- Ulasan berbahasa Indonesia informal dan pendek (6–11 kata).
- Dataset hanya memiliki 40 kalimat unik dari 200 baris, sehingga test set berukuran kecil dan hasil metrik harus diinterpretasikan dengan hati-hati.
- Gemini diakses melalui free tier, sehingga eksperimen tunduk pada rate limit API.

## 2. Import Library & Load Dataset

In [22]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
from pathlib import Path 
import pandas as pd 

DATA_PATH = Path("..")/ "data"/ "customer_reviews_sentiment.csv"

TEXT_COL = "review_text"
LABEL_COL = "sentiment"

df= pd.read_csv(DATA_PATH)
df.head ()

,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


### 2.1.a Eksplorasi Data

In [13]:
print("Jumlah baris :", df.shape[0])
print("Jumlah Kolom :", df.shape[1])
print("Nama   Kolom :", df.columns.tolist)
df.info()

Jumlah baris : 200
Jumlah Kolom : 4
Nama   Kolom : <bound method IndexOpsMixin.tolist of Index(['review_id', 'product_name', 'review_text', 'sentiment'], dtype='str')>
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   review_id     200 non-null    int64
 1   product_name  200 non-null    str  
 2   review_text   200 non-null    str  
 3   sentiment     200 non-null    str  
dtypes: int64(1), str(3)
memory usage: 6.4 KB


### 2.1.B Missing Values

In [15]:
df.isna().sum()

review_id       0
product_name    0
review_text     0
sentiment       0
dtype: int64

### 2.1.c Duplicat

In [23]:
print("Duplikat seluruh kolom    :", df.duplicated().sum())
print("Duplikat tanpa review_id  :", df.drop(columns="review_id").duplicated().sum())
print("Duplikat Review_text      :", df[TEXT_COL].duplicated().sum())
print("Jumlah Review_text unik   :", df[TEXT_COL].nunique())


label_per_text = df.groupby(TEXT_COL)[LABEL_COL].nunique()
print("Teks dengan Label Tidak Konsisten:",(label_per_text >1).sum())

Duplikat seluruh kolom    : 0
Duplikat tanpa review_id  : 31
Duplikat Review_text      : 160
Jumlah Review_text unik   : 40
Teks dengan Label Tidak Konsisten: 0


###  2.1.d: teks yang paling sering berulang

In [24]:
df[TEXT_COL].value_counts().head(10)

review_text
Bahan nyaman dipakai, ukuran pas, tidak mengecewakan.                      12
Produknya rapi dan bersih, tidak ada cacat sama sekali.                     9
Pengiriman lambat sekali, sudah dua minggu belum sampai.                    8
Kualitas produk jelek, baru dipakai sekali sudah rusak.                     8
Pelayanan ramah dan responsif, barang sampai lebih cepat dari estimasi.     8
Packing rapi, produk sesuai deskripsi. Recommended seller!                  7
Produk original dan berfungsi dengan baik, terima kasih!                    7
Sangat puas dengan pembelian ini, akan beli lagi di lain waktu.             7
Respon penjual lambat dan terkesan tidak peduli komplain.                   7
Fast respon, pengiriman cepat, barang aman sampai tujuan.                   7
Name: count, dtype: int64

### 2.1.g: karakteristik teks

In [25]:
text_stats=pd.DataFrame({
    "char_len":df[TEXT_COL].str.len(),
    "word_count":df[TEXT_COL].str.split().str.len(),
    LABEL_COL: df[LABEL_COL],
})
display(text_stats[["char_len","word_count"]].describe().round(2))
text_stats.groupby(LABEL_COL)[["char_len","word_count"]].mean().round(2)

,char_len,word_count
count,200.00,200.00
mean,55.18,7.80
std,5.14,1.11
min,43.00,6.00
25%,53.00,7.00
50%,55.50,8.00
75%,57.00,8.00
max,71.00,11.00


,char_len,word_count
sentiment,,
negatif,54.56,7.64
positif,55.70,7.94


### 2.1.g: distribusi produk

In [26]:
df["product_name"].value_counts() 

product_name
Case HP              22
Rice Cooker          18
Sepatu Sneakers      17
Jaket Hoodie         16
Tas Ransel           16
Headset Bluetooth    14
Kursi Lipat          14
Charger Cepat        14
Botol Minum          13
Kaos Polos           12
Lampu LED            11
Kemeja Flanel        10
Sandal Jepit         10
Powerbank             7
Blender Mini          6
Name: count, dtype: int64

### 2.2 Ringkasan Eksplorasi Data

- Dataset berisi **200 ulasan** dengan 4 kolom: `review_id`, `product_name`, `review_text`, `sentiment`.
- **Tidak ada missing value**, dan setiap teks memiliki label yang konsisten.
- Distribusi kelas **cukup seimbang**: positif 110 (55%) dan negatif 90 (45%). Accuracy masih bermakna, tetapi tetap dilengkapi Precision, Recall, dan F1.
- Ulasan **sangat pendek** (rata-rata 7,8 kata) dan panjangnya hampir sama antar kelas, sehingga panjang teks tidak bisa dipakai untuk membedakan sentimen.
- **Temuan kritis:** hanya **40 kalimat unik** (20 positif, 20 negatif). Setiap kalimat diulang rata-rata 5 kali (maksimum 12) dengan `product_name` berbeda. Split acak biasa akan menempatkan kalimat yang sama di train dan test, sehingga terjadi **data leakage** dan skor model klasik menggelembung.
- Beberapa ulasan positif memakai kata yang biasanya negatif, misalnya "**tidak** mengecewakan" dan "**tidak** ada **cacat**". Ini menjadi tantangan untuk model berbasis kata seperti TF-IDF.

**Implikasi untuk eksperimen:** pembagian data dilakukan berbasis grup `review_text` agar kalimat di test set tidak pernah terlihat saat training (skenario utama). Split acak biasa tetap dijalankan sebagai pembanding untuk mengukur efek leakage.

## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

In [ ]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### 4.2 Training Model

In [ ]:
clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

### 4.3 Prediksi pada Test Set

In [ ]:
y_pred_classic = clf.predict(X_test_vec)

## 5. Pendekatan 2 — LLM API (Gemini)

### 5.1 Setup API

In [ ]:
# Simpan API key di environment variable, JANGAN hardcode langsung di notebook
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

### 5.2 Merancang Prompt

_Tuliskan prompt yang kamu rancang di sini, dan jelaskan alasannya (zero-shot / few-shot)._

In [ ]:
# 🔧 Sesuaikan label pada prompt jika dataset final menggunakan label berbeda
def classify_sentiment_llm(review_text):
    prompt = f'''Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata: positif atau negatif.

Ulasan: "{review_text}"
Sentimen:'''
    response = client.models.generate_content(
        model='gemini-2.0-flash',  # sesuaikan versi model
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return response.text.strip().lower()

### 5.3 Menjalankan Prediksi pada Test Set

Catatan: lakukan normalisasi terhadap output LLM sebelum dibandingkan dengan label asli (misal: lowercase, strip whitespace).

In [ ]:
y_pred_llm = []
for review in X_test:
    pred = classify_sentiment_llm(review)
    y_pred_llm.append(pred)

# TODO: normalisasi y_pred_llm agar konsisten dengan label 'positif'/'negatif'

## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [ ]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_classic))

### 6.2 Evaluasi LLM API

In [ ]:
print('=== LLM API (Gemini) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_llm))
print(classification_report(y_test, y_pred_llm))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_llm))

### 6.3 Tabel Perbandingan Ringkasan

_Susun tabel ringkasan (bisa markdown table atau DataFrame) yang membandingkan Accuracy, Precision, Recall, F1-Score kedua pendekatan._

## 7. Analisis Trade-off dan Limitation

_Tuliskan analisis: pendekatan mana yang lebih unggul dari sisi performa, trade-off effort/kecepatan/biaya, dan keterbatasan masing-masing pendekatan._

## 8. Rekomendasi Technical Approach

_Tuliskan rekomendasi akhir beserta alasannya, berdasarkan hasil eksperimen di atas._